# 02 - Cadenas Secuenciales con LCEL

**LCEL (LangChain Expression Language)** es la forma moderna de componer pasos usando el operador `|` (pipe), igual que en una terminal Unix.

```
prompt | llm | output_parser
```

**Objetivos:**
- Entender el operador `|` para encadenar pasos
- Cadena simple: `prompt | llm`
- Cadena con parser: `prompt | llm | StrOutputParser`
- Cadena secuencial: salida de una cadena → entrada de otra
- Cadena con múltiples inputs: `RunnablePassthrough`

## 0. Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(dotenv_path="../.env")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7,
)

## 1. Cadena simple: `prompt | llm`

El output es un `AIMessage` — el objeto completo del modelo.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "Dame 3 curiosidades sobre {animal} en español."
)

# Encadenamos con |
cadena = prompt | llm

resultado = cadena.invoke({"animal": "el pulpo"})

print(type(resultado))       # AIMessage
print(resultado.content)

## 2. Con output parser: `prompt | llm | StrOutputParser`

Agrega `StrOutputParser` al final para obtener directamente un `str` en lugar de un `AIMessage`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

cadena = prompt | llm | StrOutputParser()

resultado = cadena.invoke({"animal": "el pulpo"})

print(type(resultado))   # str — ya no es AIMessage
print(resultado)

## 3. Cadena secuencial: salida de una cadena → entrada de otra

La salida de `cadena_1` se convierte en input de `cadena_2`. Útil para flujos de pasos múltiples.

```
[tema] → cadena_1 → [resumen] → cadena_2 → [pregunta de examen]
```

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Paso 1: genera un resumen del tema
prompt_resumen = ChatPromptTemplate.from_template(
    "Resume en 2 oraciones el siguiente tema: {tema}"
)

# Paso 2: genera una pregunta de examen basada en el resumen
prompt_pregunta = ChatPromptTemplate.from_template(
    "Basándote en este resumen, crea una pregunta de examen:\n\n{resumen}"
)

# Encadenamos: la salida del paso 1 pasa como {resumen} al paso 2
cadena_secuencial = (
    prompt_resumen
    | llm
    | StrOutputParser()
    | {"resumen": lambda x: x}   # empaqueta el string como dict para el siguiente prompt
    | prompt_pregunta
    | llm
    | StrOutputParser()
)

resultado = cadena_secuencial.invoke({"tema": "la inteligencia artificial generativa"})
print(resultado)

## 4. Cadena con múltiples inputs: `RunnablePassthrough`

`RunnablePassthrough` pasa el input original sin modificarlo al siguiente paso, permitiendo que una cadena reciba tanto el input original como resultados intermedios.

```
{tema} ──────────────────────────────→ |
{tema} → cadena_resumen → {resumen} → | → prompt_final → llm
```

In [ ]:
from langchain_core.runnables import RunnablePassthrough

cadena_resumen = prompt_resumen | llm | StrOutputParser()

prompt_final = ChatPromptTemplate.from_template(
    "Tema original: {tema}\n\nResumen: {resumen}\n\n¿Qué tan bien cubre el resumen el tema original? Evalúa del 1 al 10."
)

cadena_con_passthrough = (
    {"tema": RunnablePassthrough(), "resumen": cadena_resumen}
    | prompt_final
    | llm
    | StrOutputParser()
)

resultado = cadena_con_passthrough.invoke("la fotosíntesis")
print(resultado)

## 5. Comparación: `PromptTemplate` vs `ChatPromptTemplate`

`PromptTemplate` produce un string plano. Para usarlo con un modelo de chat como Gemini, LangChain lo convierte internamente en un `HumanMessage`. Es útil cuando el prompt es simple y no necesitas roles.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
learning_prompt = PromptTemplate(
    input_variables=["activity"],
    template="I want to learn how to {activity}. Can you suggest how I can learn this step-by-step?"
)

time_prompt = PromptTemplate(
    input_variables=["learning_plan"],
    template="I only have one week. Can you create a concise plan to help me hit this goal: {learning_plan}."
)

# Complete the sequential chain with LCEL
seq_chain = ({"learning_plan": learning_prompt | llm | StrOutputParser()}
    | time_prompt
    | llm
    | StrOutputParser())

# Call the chain
print(seq_chain.invoke({"activity": "play chess"}))

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- CON PromptTemplate ---
# Genera un string plano, sin roles
prompt_plano = PromptTemplate.from_template(
    "Dame 3 curiosidades sobre {animal} en español."
)

cadena_plana = prompt_plano | llm | StrOutputParser()

# Inspecciona lo que genera el prompt ANTES de enviarlo al modelo
print("=== PromptTemplate ===")
print(type(prompt_plano.invoke({"animal": "el pulpo"})))  # StringPromptValue
print(prompt_plano.invoke({"animal": "el pulpo"}).text)

print("\nRespuesta:", cadena_plana.invoke({"animal": "el pulpo"}))

# --- CON ChatPromptTemplate ---
# Genera una lista de mensajes con roles
prompt_chat = ChatPromptTemplate.from_template(
    "Dame 3 curiosidades sobre {animal} en español."
)

print("\n=== ChatPromptTemplate ===")
print(type(prompt_chat.invoke({"animal": "el pulpo"})))   # ChatPromptValue
print(prompt_chat.invoke({"animal": "el pulpo"}).messages)